In [ ]:
import numpy as np  # Imports the NumPy library, typically used for scientific and numerical array operations

from analysis import analyze_frame, demo_frame  # Imports the framework's core analysis and mock-data generation functions


def test_demo_analysis_returns_comparable_models_and_forecasts():  # Defines a test case to verify that a valid dataset returns correct models and outputs
    result = analyze_frame(demo_frame(rows=120), symbol="DEMO")  # Generates 120 rows of mock data and passes it to the analyzer with the ticker "DEMO"
    assert result["symbol"] == "DEMO"  # Assets (verifies) that the returned analysis metadata correctly keeps the "DEMO" symbol name
    assert result["observations"] >= 80  # Asserts that the total count of processed historical data points is at least 80
    assert set(result["models"]) == {"Linear Regression", "Random Forest", "ARIMA", "LSTM"}  # Asserts that the keys in the models dictionary match these exact four machine learning/statistical models
    assert all(len(model["forecast"]) == 30 for model in result["models"].values())  # Asserts that every single model generated a prediction horizon of exactly 30 periods
    assert all(model["metrics"]["rmse"] is not None for model in result["models"].values())  # Asserts that every model successfully computed an RMSE (Root Mean Squared Error) value instead of returning None
    assert len(result["correlation"]) == 4  # Asserts that the cross-model correlation matrix contains data dimensions for all 4 models


def test_analysis_rejects_short_series():  # Defines a test case to ensure the system gracefully handles and rejects insufficient data inputs
    frame = demo_frame(rows=30)  # Generates an intentionally short dataset consisting of only 30 rows
    try:  # Begins a try-except block to safely capture an anticipated validation failure
        analyze_frame(frame)  # Attempts to execute the analysis on the short data, expecting it to trigger a data validation rule
    except ValueError as error:  # Catches the expected ValueError exception and references it as the variable 'error'
        assert "80" in str(error)  # Asserts that the exception text contains the substring "80", confirming it informs the user of the minimum required length
    else:  # Executes only if the code inside the try block completed successfully without throwing any exceptions
        raise AssertionError("Expected short input to be rejected")  # Explicitly fails the test since the analyzer should never accept a 30-row dataset
